## 1. Load libraries and task folders

Read the task-specific train/test matrices, labels, metadata, and selected peptide feature lists created in `preprocessing.ipynb`.


In [10]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

warnings.filterwarnings("ignore", category=ConvergenceWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)


In [11]:
# Same process as in the Logistic Regression baseline model

PROCESSED_DIR = Path("data/processed")
TASK_NAMES = [
    "healthy_vs_infected",
    "symptomatic_non_covid_vs_covid",
    "severe_vs_nonsevere",
]

task_data = {}

for task_name in TASK_NAMES:
    task_dir = PROCESSED_DIR / task_name
    task_data[task_name] = {
        "dir": task_dir,
        "X_train": pd.read_csv(task_dir / "X_train_scaled.csv", index_col=0),
        "X_test": pd.read_csv(task_dir / "X_test_scaled.csv", index_col=0),
        "y_train": pd.read_csv(task_dir / "y_train.csv", index_col=0).squeeze("columns"),
        "y_test": pd.read_csv(task_dir / "y_test.csv", index_col=0).squeeze("columns"),
        "metadata_train": pd.read_csv(task_dir / "metadata_train.csv", index_col=0),
        "metadata_test": pd.read_csv(task_dir / "metadata_test.csv", index_col=0),
        "selected_peptides": pd.read_csv(task_dir / "selected_peptides.csv"),
    }

    print(f"\n{task_name}")
    print(f"X_train shape: {task_data[task_name]['X_train'].shape}")
    print(f"X_test shape: {task_data[task_name]['X_test'].shape}")
    print("Train label counts:")
    print(task_data[task_name]["metadata_train"]["label_name"].value_counts())
    print("Test label counts:")
    print(task_data[task_name]["metadata_test"]["label_name"].value_counts())



healthy_vs_infected
X_train shape: (72, 35882)
X_test shape: (18, 35882)
Train label counts:
label_name
Infected    54
Healthy     18
Name: count, dtype: int64
Test label counts:
label_name
Infected    14
Healthy      4
Name: count, dtype: int64

symptomatic_non_covid_vs_covid
X_train shape: (54, 35882)
X_test shape: (14, 35882)
Train label counts:
label_name
COVID-19                    34
Symptomatic-non-COVID-19    20
Name: count, dtype: int64
Test label counts:
label_name
COVID-19                    9
Symptomatic-non-COVID-19    5
Name: count, dtype: int64

severe_vs_nonsevere
X_train shape: (34, 35882)
X_test shape: (9, 35882)
Train label counts:
label_name
Non-severe-COVID-19    20
Severe-COVID-19        14
Name: count, dtype: int64
Test label counts:
label_name
Non-severe-COVID-19    5
Severe-COVID-19        4
Name: count, dtype: int64


## Random-choice baseline 

Classifies the input randomly, where the chance of selecting each group is equal to that group's proportion in the data. Averaged over 1000 runs for stable estimates.

In [12]:
N_BASELINE_RUNS = 1000
baseline_rows = []

for task_name, data in task_data.items():
    print(f"Running {task_name}...")

    X_train = data["X_train"]
    y_train = data["y_train"]
    X_test = data["X_test"]
    y_test = data["y_test"]
    y_all = pd.concat([y_train, y_test])
    prevalence = float(y_all.mean())

    run_metrics = {
        "accuracy":          [],
        "balanced_accuracy": [],
        "precision":         [],
        "recall":            [],
        "f1":                [],
    }
    # roc_auc and auprc are deterministic for a stratified dummy — predict_proba
    # always returns fixed class proportions as constant scores regardless of which
    # labels were randomly drawn — so those are hardcoded to their theoretical values

    for seed in range(N_BASELINE_RUNS):
        # Stratified dummy: generates random predictions by respecting the training set's class distribution
        dummy = DummyClassifier(strategy="stratified", random_state=seed)
        dummy.fit(X_train, y_train) # doesn't actually depend on X
        y_pred = dummy.predict(X_test)

        run_metrics["accuracy"].append(accuracy_score(y_test, y_pred))
        run_metrics["balanced_accuracy"].append(balanced_accuracy_score(y_test, y_pred))
        run_metrics["precision"].append(precision_score(y_test, y_pred, zero_division=0))
        run_metrics["recall"].append(recall_score(y_test, y_pred, zero_division=0))
        run_metrics["f1"].append(f1_score(y_test, y_pred, zero_division=0))

    baseline_rows.append({
        "task": task_name,
        "positive_prevalence": prevalence,
        "test_accuracy": np.mean(run_metrics["accuracy"]),
        "test_balanced_accuracy": np.mean(run_metrics["balanced_accuracy"]),
        "test_precision": np.mean(run_metrics["precision"]),
        "test_recall": np.mean(run_metrics["recall"]),
        "test_f1": np.mean(run_metrics["f1"]),
        "test_roc_auc": 0.5,
        "test_auprc": prevalence,  # random classifier AUPRC = prevalence
    })

random_df = pd.DataFrame(baseline_rows)
random_df = random_df.sort_values(
    by=["task", "test_balanced_accuracy"],
    ascending=[True, False],
).reset_index(drop=True)
random_df

Running healthy_vs_infected...
Running symptomatic_non_covid_vs_covid...
Running severe_vs_nonsevere...


,task,positive_prevalence,test_accuracy,test_balanced_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_auprc
0,healthy_vs_infected,0.755556,0.640000,0.500982,0.778131,0.751214,0.760635,0.5,0.755556
1,severe_vs_nonsevere,0.418605,0.514556,0.504550,0.440444,0.414500,0.410312,0.5,0.418605
2,symptomatic_non_covid_vs_covid,0.632353,0.539214,0.502144,0.644541,0.631889,0.630788,0.5,0.632353


## 5. Save results

In [13]:
results_path = PROCESSED_DIR / "random_choice_baseline_results_all_tasks.csv"
random_df.to_csv(results_path, index=False)
print(f"Saved combined results table to: {results_path.resolve()}")


Saved combined results table to: /Users/lexiestucki/Desktop/291B/cse291-project/data/processed/random_choice_baseline_results_all_tasks.csv
